# Data Processing

This notebook cleans and preprocesses the Global Air Quality and Beijing Multi-Site Air Quality datasets by removing unnecessary columns, missing values, duplicates, and invalid measurements. It also performs one-hot encoding and creates three PM2.5 time-series forecasting targets (1, 6, and 24 hours ahead) by matching future measurements using timestamps within each city or station. Both datasets are split chronologically into training and testing sets to prevent data leakage.

**Important Notes:** Normalization and feature selection are left to individual model implementations. To prevent data leakage, perform feature selection and fit normalization using **training data only**, then apply the same transformations to the test data. All processed datasets have been exported to the `processed/` folder, so please use these files directly instead of repeating preprocessing.

**Model Implementation:** Each modeling technique should be trained separately on the Global and Beijing datasets using the same exported splits and evaluation metrics (R², RMSE, MAE). This allows us to compare which models forecast PM2.5 most accurately and whether their performance is more consistent across geographically diverse cities or monitoring stations within one region.

In [1]:
import pandas as pd
import os

## Load Datasets

In [2]:
# load global air quality dataset
global_air = pd.read_csv("datasets/Global_Air_Quality.csv")

# load all 12 beijing station datasets
beijing_files = [
    "Aotizhongxin.csv",
    "Changping.csv",
    "Dingling.csv",
    "Dongsi.csv",
    "Guanyuan.csv",
    "Gucheng.csv",
    "Huairou.csv",
    "Nongzhanguan.csv",
    "Shunyi.csv",
    "Tiantan.csv",
    "Wanliu.csv",
    "Wanshouxigong.csv"
]

# combine beijing datasets into one dataframe
beijing_air = pd.concat(
    [pd.read_csv("datasets/" + file) for file in beijing_files],
    ignore_index=True
)

# display dataset dimensions
print("Global Air Quality:", global_air.shape)
print("Beijing Air Quality:", beijing_air.shape)

# preview datasets
print("\nGlobal Air Quality:")
display(global_air.head())
print("\nBeijing Air Quality:")
display(beijing_air.head())

Global Air Quality: (52560, 9)
Beijing Air Quality: (420768, 18)

Global Air Quality:


,Date,City,CO,NO2,SO2,O3,PM2.5,PM10,AQI
0,2023-01-01 00:00:00+00:00,Brasilia,292.0,30.15,1.10,23.0,11.1,15.8,21.158330
1,2023-01-01 01:00:00+00:00,Brasilia,291.0,28.95,1.15,24.0,12.4,17.7,20.825000
2,2023-01-01 02:00:00+00:00,Brasilia,289.0,27.25,1.25,25.0,13.0,18.5,20.474998
3,2023-01-01 03:00:00+00:00,Brasilia,275.0,23.70,1.30,28.0,9.2,13.1,20.074999
4,2023-01-01 04:00:00+00:00,Brasilia,246.0,19.35,1.25,32.0,6.8,9.7,19.324999



Beijing Air Quality:


,No,year,month,day,hour,PM2.5,PM10,SO2,NO2,CO,O3,TEMP,PRES,DEWP,RAIN,wd,WSPM,station
0,1,2013,3,1,0,4.0,4.0,4.0,7.0,300.0,77.0,-0.7,1023.0,-18.8,0.0,NNW,4.4,Aotizhongxin
1,2,2013,3,1,1,8.0,8.0,4.0,7.0,300.0,77.0,-1.1,1023.2,-18.2,0.0,N,4.7,Aotizhongxin
2,3,2013,3,1,2,7.0,7.0,5.0,10.0,300.0,73.0,-1.1,1023.5,-18.2,0.0,NNW,5.6,Aotizhongxin
3,4,2013,3,1,3,6.0,6.0,11.0,11.0,300.0,72.0,-1.4,1024.5,-19.4,0.0,NW,3.1,Aotizhongxin
4,5,2013,3,1,4,3.0,3.0,12.0,12.0,300.0,72.0,-2.0,1025.2,-19.5,0.0,N,2.0,Aotizhongxin


## Data Exploration

In [3]:
# display dataset dimensions
print("Global Air Quality Shape:", global_air.shape)
print("Beijing Air Quality Shape:", beijing_air.shape)

# display column names
print("\nGlobal Air Quality Columns:")
print(global_air.columns.tolist())
print("\nBeijing Air Quality Columns:")
print(beijing_air.columns.tolist())

# display dataset information
print("\nGlobal Air Quality Information:")
global_air.info()
print("\nBeijing Air Quality Information:")
beijing_air.info()

Global Air Quality Shape: (52560, 9)
Beijing Air Quality Shape: (420768, 18)

Global Air Quality Columns:
['Date', 'City', 'CO', 'NO2', 'SO2', 'O3', 'PM2.5', 'PM10', 'AQI']

Beijing Air Quality Columns:
['No', 'year', 'month', 'day', 'hour', 'PM2.5', 'PM10', 'SO2', 'NO2', 'CO', 'O3', 'TEMP', 'PRES', 'DEWP', 'RAIN', 'wd', 'WSPM', 'station']

Global Air Quality Information:
<class 'pandas.DataFrame'>
RangeIndex: 52560 entries, 0 to 52559
Data columns (total 9 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Date    52560 non-null  str    
 1   City    52560 non-null  str    
 2   CO      52560 non-null  float64
 3   NO2     52560 non-null  float64
 4   SO2     52560 non-null  float64
 5   O3      52560 non-null  float64
 6   PM2.5   52560 non-null  float64
 7   PM10    52560 non-null  float64
 8   AQI     52560 non-null  float64
dtypes: float64(7), str(2)
memory usage: 3.6 MB

Beijing Air Quality Information:
<class 'pandas.DataFrame'>
RangeIndex: 

In [4]:
# count missing values for each column
print("Global Air Quality Missing Values:")
print(global_air.isnull().sum())
print("\nBeijing Air Quality Missing Values:")
print(beijing_air.isnull().sum())

Global Air Quality Missing Values:
Date     0
City     0
CO       0
NO2      0
SO2      0
O3       0
PM2.5    0
PM10     0
AQI      0
dtype: int64

Beijing Air Quality Missing Values:
No             0
year           0
month          0
day            0
hour           0
PM2.5       8739
PM10        6449
SO2         9021
NO2        12116
CO         20701
O3         13277
TEMP         398
PRES         393
DEWP         403
RAIN         390
wd          1822
WSPM         318
station        0
dtype: int64


In [5]:
# count rows with missing pollutant values
pollutants = ["PM2.5", "PM10", "SO2", "NO2", "CO", "O3"]

missing_rows = beijing_air[pollutants].isnull().any(axis=1).sum()

print("Total Rows:", len(beijing_air))
print("Rows to Remove:", missing_rows)
print("Remaining Rows:", len(beijing_air) - missing_rows)

Total Rows: 420768
Rows to Remove: 36788
Remaining Rows: 383980


In [6]:
# count duplicate rows
print("Global Air Quality Duplicates:", global_air.duplicated().sum())
print("Beijing Air Quality Duplicates:", beijing_air.duplicated().sum())

Global Air Quality Duplicates: 0
Beijing Air Quality Duplicates: 0


In [7]:
# display summary statistics for numerical columns
print("Global Air Quality Summary Statistics:")
display(global_air.describe())
print("\nBeijing Air Quality Summary Statistics:")
display(beijing_air.describe())

Global Air Quality Summary Statistics:


,CO,NO2,SO2,O3,PM2.5,PM10,AQI
count,52560.000000,52560.000000,52560.000000,52560.000000,52560.000000,52560.000000,52560.000000
mean,273.372565,28.930514,12.051645,55.304433,18.621792,35.505850,41.941352
std,171.578815,25.473936,14.684127,36.939938,15.897499,41.866614,24.989747
min,48.000000,0.000000,0.000000,0.000000,0.000000,0.100000,8.000000
25%,160.000000,11.100000,2.500000,29.000000,7.100000,11.000000,22.638333
50%,224.000000,22.100000,5.800000,51.000000,13.400000,20.800000,32.400000
75%,335.000000,38.700000,17.300000,74.000000,24.900000,41.700000,61.013332
max,2145.000000,251.900000,200.800000,425.000000,143.400000,550.000000,188.319980



Beijing Air Quality Summary Statistics:


,No,year,month,day,hour,PM2.5,PM10,SO2,NO2,CO,O3,TEMP,PRES,DEWP,RAIN,WSPM
count,420768.000000,420768.000000,420768.000000,420768.000000,420768.000000,412029.000000,414319.000000,411747.000000,408652.000000,400067.000000,407491.000000,420370.000000,420375.000000,420365.000000,420378.000000,420450.000000
mean,17532.500000,2014.662560,6.522930,15.729637,11.500000,79.793428,104.602618,15.830835,50.638586,1230.766454,57.372271,13.538976,1010.746982,2.490822,0.064476,1.729711
std,10122.116943,1.177198,3.448707,8.800102,6.922195,80.822391,91.772426,21.650603,35.127912,1160.182716,56.661607,11.436139,10.474055,13.793847,0.821004,1.246386
min,1.000000,2013.000000,1.000000,1.000000,0.000000,2.000000,2.000000,0.285600,1.026500,100.000000,0.214200,-19.900000,982.400000,-43.400000,0.000000,0.000000
25%,8766.750000,2014.000000,4.000000,8.000000,5.750000,20.000000,36.000000,3.000000,23.000000,500.000000,11.000000,3.100000,1002.300000,-8.900000,0.000000,0.900000
50%,17532.500000,2015.000000,7.000000,16.000000,11.500000,55.000000,82.000000,7.000000,43.000000,900.000000,45.000000,14.500000,1010.400000,3.100000,0.000000,1.400000
75%,26298.250000,2016.000000,10.000000,23.000000,17.250000,111.000000,145.000000,20.000000,71.000000,1500.000000,82.000000,23.300000,1019.000000,15.100000,0.000000,2.200000
max,35064.000000,2017.000000,12.000000,31.000000,23.000000,999.000000,999.000000,500.000000,290.000000,10000.000000,1071.000000,41.600000,1042.800000,29.100000,72.500000,13.200000


In [8]:
# examine observations across cities and stations
print("Global Air Quality Observations by City:")
print(global_air["City"].value_counts())
print("\nBeijing Air Quality Observations by Station:")
print(beijing_air["station"].value_counts())

Global Air Quality Observations by City:
City
Brasilia    8760
Cairo       8760
Dubai       8760
London      8760
New York    8760
Sydney      8760
Name: count, dtype: int64

Beijing Air Quality Observations by Station:
station
Aotizhongxin     35064
Changping        35064
Dingling         35064
Dongsi           35064
Guanyuan         35064
Gucheng          35064
Huairou          35064
Nongzhanguan     35064
Shunyi           35064
Tiantan          35064
Wanliu           35064
Wanshouxigong    35064
Name: count, dtype: int64


## Data Cleaning

In [9]:
# create copies of original datasets
global_clean = global_air.copy()
beijing_clean = beijing_air.copy()

In [10]:
# remove unnecessary identifier columns
beijing_clean = beijing_clean.drop(columns=["No"], errors="ignore")

# remove AQI derived from pollutant measurements
global_clean = global_clean.drop(columns=["AQI"])

In [11]:
# convert global dates to datetime
global_clean["Date"] = pd.to_datetime(global_clean["Date"], errors="coerce", utc=True)

# combine beijing date and time columns
beijing_clean["Date"] = pd.to_datetime(
    beijing_clean[["year", "month", "day", "hour"]],
    errors="coerce"
)

# display timestamps
display(global_clean[["Date", "City"]].head())
display(beijing_clean[["Date", "station"]].head())

,Date,City
0,2023-01-01 00:00:00+00:00,Brasilia
1,2023-01-01 01:00:00+00:00,Brasilia
2,2023-01-01 02:00:00+00:00,Brasilia
3,2023-01-01 03:00:00+00:00,Brasilia
4,2023-01-01 04:00:00+00:00,Brasilia


,Date,station
0,2013-03-01 00:00:00,Aotizhongxin
1,2013-03-01 01:00:00,Aotizhongxin
2,2013-03-01 02:00:00,Aotizhongxin
3,2013-03-01 03:00:00,Aotizhongxin
4,2013-03-01 04:00:00,Aotizhongxin


In [12]:
# count duplicate location and timestamp records
print("Global Duplicates:", global_clean.duplicated(subset=["City", "Date"]).sum())
print("Beijing Duplicates:", beijing_clean.duplicated(subset=["station", "Date"]).sum())

# remove duplicate observations
global_clean = global_clean.drop_duplicates(subset=["City", "Date"])
beijing_clean = beijing_clean.drop_duplicates(subset=["station", "Date"])

Global Duplicates: 0
Beijing Duplicates: 0


In [13]:
# count rows containing missing values
print("Global Missing Rows:", global_clean.isnull().any(axis=1).sum())
print("Beijing Missing Rows:", beijing_clean.isnull().any(axis=1).sum())

# remove rows with missing values
global_clean = global_clean.dropna().copy()
beijing_clean = beijing_clean.dropna().copy()

# display remaining observations
print("\nGlobal Remaining Rows:", len(global_clean))
print("Beijing Remaining Rows:", len(beijing_clean))

Global Missing Rows: 0
Beijing Missing Rows: 38600

Global Remaining Rows: 52560
Beijing Remaining Rows: 382168


In [14]:
# common pollutant columns
pollutants = ["PM2.5", "PM10", "SO2", "NO2", "CO", "O3"]

# remove rows with negative pollutant measurements
global_clean = global_clean[
    (global_clean[pollutants] >= 0).all(axis=1)
].copy()

beijing_clean = beijing_clean[
    (beijing_clean[pollutants] >= 0).all(axis=1)
].copy()

In [15]:
# sort observations by location and time
global_clean = global_clean.sort_values(["City", "Date"]).reset_index(drop=True)
beijing_clean = beijing_clean.sort_values(["station", "Date"]).reset_index(drop=True)

In [16]:
# display dataset dimensions
print("Global Dataset Shape:", global_clean.shape)
print("Beijing Dataset Shape:", beijing_clean.shape)

# display column names
print("\nGlobal Columns:")
print(global_clean.columns.tolist())
print("\nBeijing Columns:")
print(beijing_clean.columns.tolist())

# count observations by location
print("\nGlobal Observations by City:")
print(global_clean["City"].value_counts())
print("\nBeijing Observations by Station:")
print(beijing_clean["station"].value_counts())

# preview both datasets
print("\nGlobal Air Quality:")
display(global_clean.head())
print("\nBeijing Air Quality:")
display(beijing_clean.head())

Global Dataset Shape: (52560, 8)
Beijing Dataset Shape: (382168, 18)

Global Columns:
['Date', 'City', 'CO', 'NO2', 'SO2', 'O3', 'PM2.5', 'PM10']

Beijing Columns:
['year', 'month', 'day', 'hour', 'PM2.5', 'PM10', 'SO2', 'NO2', 'CO', 'O3', 'TEMP', 'PRES', 'DEWP', 'RAIN', 'wd', 'WSPM', 'station', 'Date']

Global Observations by City:
City
Brasilia    8760
Cairo       8760
Dubai       8760
London      8760
New York    8760
Sydney      8760
Name: count, dtype: int64

Beijing Observations by Station:
station
Nongzhanguan     33114
Tiantan          32843
Wanshouxigong    32768
Changping        32681
Gucheng          32504
Guanyuan         32263
Aotizhongxin     31815
Huairou          31708
Dingling         31306
Wanliu           30634
Dongsi           30338
Shunyi           30194
Name: count, dtype: int64

Global Air Quality:


,Date,City,CO,NO2,SO2,O3,PM2.5,PM10
0,2023-01-01 00:00:00+00:00,Brasilia,292.0,30.15,1.10,23.0,11.1,15.8
1,2023-01-01 01:00:00+00:00,Brasilia,291.0,28.95,1.15,24.0,12.4,17.7
2,2023-01-01 02:00:00+00:00,Brasilia,289.0,27.25,1.25,25.0,13.0,18.5
3,2023-01-01 03:00:00+00:00,Brasilia,275.0,23.70,1.30,28.0,9.2,13.1
4,2023-01-01 04:00:00+00:00,Brasilia,246.0,19.35,1.25,32.0,6.8,9.7



Beijing Air Quality:


,year,month,day,hour,PM2.5,PM10,SO2,NO2,CO,O3,TEMP,PRES,DEWP,RAIN,wd,WSPM,station,Date
0,2013,3,1,0,4.0,4.0,4.0,7.0,300.0,77.0,-0.7,1023.0,-18.8,0.0,NNW,4.4,Aotizhongxin,2013-03-01 00:00:00
1,2013,3,1,1,8.0,8.0,4.0,7.0,300.0,77.0,-1.1,1023.2,-18.2,0.0,N,4.7,Aotizhongxin,2013-03-01 01:00:00
2,2013,3,1,2,7.0,7.0,5.0,10.0,300.0,73.0,-1.1,1023.5,-18.2,0.0,NNW,5.6,Aotizhongxin,2013-03-01 02:00:00
3,2013,3,1,3,6.0,6.0,11.0,11.0,300.0,72.0,-1.4,1024.5,-19.4,0.0,NW,3.1,Aotizhongxin,2013-03-01 03:00:00
4,2013,3,1,4,3.0,3.0,12.0,12.0,300.0,72.0,-2.0,1025.2,-19.5,0.0,N,2.0,Aotizhongxin,2013-03-01 04:00:00


## Data Encoding 

In [17]:
# create copies of cleaned datasets
global_processed = global_clean.copy()
beijing_processed = beijing_clean.copy()

In [18]:
# display categorical columns
print("Global Categorical Features:")
print(global_processed.select_dtypes(include="str").columns.tolist())
print("\nBeijing Categorical Features:")
print(beijing_processed.select_dtypes(include="str").columns.tolist())

# examine wind direction categories
print("\nBeijing Wind Directions:")
print(beijing_processed["wd"].value_counts())

Global Categorical Features:
['City']

Beijing Categorical Features:
['wd', 'station']

Beijing Wind Directions:
wd
NE     39319
ENE    31139
NW     30042
N      27994
E      27296
SW     26356
NNE    25596
NNW    23152
ESE    22188
WNW    22156
SSW    20696
SE     18365
WSW    18210
S      17683
SSE    16057
W      15919
Name: count, dtype: int64


In [19]:
# convert wind direction into numerical columns
beijing_processed = pd.get_dummies(
    beijing_processed,
    columns=["wd"],
    dtype=int
)

# display encoded wind direction columns
wind_columns = [
    col for col in beijing_processed.columns
    if col.startswith("wd_")
]

print("Encoded Wind Direction Features:")
print(wind_columns)
display(beijing_processed[wind_columns].head())

Encoded Wind Direction Features:
['wd_E', 'wd_ENE', 'wd_ESE', 'wd_N', 'wd_NE', 'wd_NNE', 'wd_NNW', 'wd_NW', 'wd_S', 'wd_SE', 'wd_SSE', 'wd_SSW', 'wd_SW', 'wd_W', 'wd_WNW', 'wd_WSW']


,wd_E,wd_ENE,wd_ESE,wd_N,wd_NE,wd_NNE,wd_NNW,wd_NW,wd_S,wd_SE,wd_SSE,wd_SSW,wd_SW,wd_W,wd_WNW,wd_WSW
0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0
1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0
4,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0


In [20]:
# columns used for identification and timestamps
global_metadata = ["Date", "City", "AQI"]
beijing_metadata = ["Date", "station", "year", "month", "day", "hour"]

# identify input features for both datasets
global_features = [
    col for col in global_processed.columns
    if col not in global_metadata
]
beijing_features = [
    col for col in beijing_processed.columns
    if col not in beijing_metadata
]

print("Global Input Features:", global_features)
print("Beijing Input Features:", beijing_features)
print("\nGlobal Input Count:", len(global_features))
print("Beijing Input Count:", len(beijing_features))

Global Input Features: ['CO', 'NO2', 'SO2', 'O3', 'PM2.5', 'PM10']
Beijing Input Features: ['PM2.5', 'PM10', 'SO2', 'NO2', 'CO', 'O3', 'TEMP', 'PRES', 'DEWP', 'RAIN', 'WSPM', 'wd_E', 'wd_ENE', 'wd_ESE', 'wd_N', 'wd_NE', 'wd_NNE', 'wd_NNW', 'wd_NW', 'wd_S', 'wd_SE', 'wd_SSE', 'wd_SSW', 'wd_SW', 'wd_W', 'wd_WNW', 'wd_WSW']

Global Input Count: 6
Beijing Input Count: 27


In [21]:
# display dataset dimensions
print("Global Processed Shape:", global_processed.shape)
print("Beijing Processed Shape:", beijing_processed.shape)

# check remaining missing values
print("\nGlobal Missing Values:", global_processed.isnull().sum().sum())
print("Beijing Missing Values:", beijing_processed.isnull().sum().sum())

# display number of input features
print("\nGlobal Input Features:", len(global_features))
print("Beijing Input Features:", len(beijing_features))

# preview processed datasets
print("\nGlobal Air Quality:")
display(global_processed.head())
print("\nBeijing Air Quality:")
display(beijing_processed.head())

Global Processed Shape: (52560, 8)
Beijing Processed Shape: (382168, 33)

Global Missing Values: 0
Beijing Missing Values: 0

Global Input Features: 6
Beijing Input Features: 27

Global Air Quality:


,Date,City,CO,NO2,SO2,O3,PM2.5,PM10
0,2023-01-01 00:00:00+00:00,Brasilia,292.0,30.15,1.10,23.0,11.1,15.8
1,2023-01-01 01:00:00+00:00,Brasilia,291.0,28.95,1.15,24.0,12.4,17.7
2,2023-01-01 02:00:00+00:00,Brasilia,289.0,27.25,1.25,25.0,13.0,18.5
3,2023-01-01 03:00:00+00:00,Brasilia,275.0,23.70,1.30,28.0,9.2,13.1
4,2023-01-01 04:00:00+00:00,Brasilia,246.0,19.35,1.25,32.0,6.8,9.7



Beijing Air Quality:


,year,month,day,hour,PM2.5,PM10,SO2,NO2,CO,O3,...,wd_NNW,wd_NW,wd_S,wd_SE,wd_SSE,wd_SSW,wd_SW,wd_W,wd_WNW,wd_WSW
0,2013,3,1,0,4.0,4.0,4.0,7.0,300.0,77.0,...,1,0,0,0,0,0,0,0,0,0
1,2013,3,1,1,8.0,8.0,4.0,7.0,300.0,77.0,...,0,0,0,0,0,0,0,0,0,0
2,2013,3,1,2,7.0,7.0,5.0,10.0,300.0,73.0,...,1,0,0,0,0,0,0,0,0,0
3,2013,3,1,3,6.0,6.0,11.0,11.0,300.0,72.0,...,0,1,0,0,0,0,0,0,0,0
4,2013,3,1,4,3.0,3.0,12.0,12.0,300.0,72.0,...,0,0,0,0,0,0,0,0,0,0


## Target Variables

In [22]:
# define target column names
target_columns = [
    "PM25_1hr_ahead",
    "PM25_6hr_ahead",
    "PM25_24hr_ahead"
]

In [23]:
# create future PM2.5 target variables (time-series forecasting)
def create_targets(df, location_col):
    df = df.copy()

    # sort observations by location and date
    df = df.sort_values([location_col, "Date"]).reset_index(drop=True)

    # create targets for 1, 6, and 24 hours ahead
    for hour in [1, 6, 24]:

        # get future PM2.5 measurements
        future = df[[location_col, "Date", "PM2.5"]].copy()

        # shift future timestamps backward to match current observations
        future["Date"] = future["Date"] - pd.Timedelta(hours=hour)

        # rename PM2.5 as the future target
        future = future.rename(
            columns={"PM2.5": f"PM25_{hour}hr_ahead"}
        )

        # match future measurements within the same location
        df = df.merge(
            future,
            on=[location_col, "Date"],
            how="left",
            validate="one_to_one"
        )

    return df

In [24]:
# create future targets for Global Air Quality
global_targets = create_targets(
    global_processed, "City"
)

# create future targets for Beijing Air Quality
beijing_targets = create_targets(
    beijing_processed, "station"
)

# display dataset dimensions
print("Global Shape:", global_targets.shape)
print("Beijing Shape:", beijing_targets.shape)


Global Shape: (52560, 11)
Beijing Shape: (382168, 36)


In [25]:
# count rows with missing target values
print("Global Rows With Missing Targets:", global_targets[target_columns].isnull().any(axis=1).sum())
print("Beijing Rows With Missing Targets:", beijing_targets[target_columns].isnull().any(axis=1).sum())

# remove rows with missing target values
global_targets = global_targets.dropna(
    subset=target_columns
).reset_index(drop=True)
beijing_targets = beijing_targets.dropna(
    subset=target_columns
).reset_index(drop=True)

# display remaining observations
print("\nGlobal Remaining Rows:", len(global_targets))
print("Beijing Remaining Rows:", len(beijing_targets))

Global Rows With Missing Targets: 144
Beijing Rows With Missing Targets: 42425

Global Remaining Rows: 52416
Beijing Remaining Rows: 339743


In [26]:
# display target column names
print("Target Variables:", target_columns)

# verify missing target values
print("\nGlobal Missing Targets:")
print(global_targets[target_columns].isnull().sum())
print("\nBeijing Missing Targets:")
print(beijing_targets[target_columns].isnull().sum())

# display target statistics
print("\nGlobal Target Statistics:")
display(global_targets[target_columns].describe())
print("\nBeijing Target Statistics:")
display(beijing_targets[target_columns].describe())

# preview targets with current PM2.5 measurements
print("\nGlobal Air Quality Targets:")
display(
    global_targets[
        ["City", "Date", "PM2.5"] + target_columns
    ].head(10)
)

print("\nBeijing Air Quality Targets:")
display(
    beijing_targets[
        ["station", "Date", "PM2.5"] + target_columns
    ].head(10)
)

Target Variables: ['PM25_1hr_ahead', 'PM25_6hr_ahead', 'PM25_24hr_ahead']

Global Missing Targets:
PM25_1hr_ahead     0
PM25_6hr_ahead     0
PM25_24hr_ahead    0
dtype: int64

Beijing Missing Targets:
PM25_1hr_ahead     0
PM25_6hr_ahead     0
PM25_24hr_ahead    0
dtype: int64

Global Target Statistics:


,PM25_1hr_ahead,PM25_6hr_ahead,PM25_24hr_ahead
count,52416.000000,52416.000000,52416.000000
mean,18.631967,18.634579,18.629117
std,15.904398,15.908975,15.906097
min,0.000000,0.000000,0.000000
25%,7.100000,7.100000,7.100000
50%,13.400000,13.400000,13.400000
75%,24.900000,24.900000,24.900000
max,143.400000,143.400000,143.400000



Beijing Target Statistics:


,PM25_1hr_ahead,PM25_6hr_ahead,PM25_24hr_ahead
count,339743.000000,339743.000000,339743.000000
mean,79.237249,79.408980,79.297605
std,79.272931,79.486538,79.594244
min,2.000000,2.000000,2.000000
25%,20.000000,21.000000,20.000000
50%,55.000000,55.000000,55.000000
75%,110.000000,111.000000,110.000000
max,844.000000,844.000000,844.000000



Global Air Quality Targets:


,City,Date,PM2.5,PM25_1hr_ahead,PM25_6hr_ahead,PM25_24hr_ahead
0,Brasilia,2023-01-01 00:00:00+00:00,11.1,12.4,5.1,7.0
1,Brasilia,2023-01-01 01:00:00+00:00,12.4,13.0,4.9,6.8
2,Brasilia,2023-01-01 02:00:00+00:00,13.0,9.2,4.9,6.6
3,Brasilia,2023-01-01 03:00:00+00:00,9.2,6.8,4.9,7.1
4,Brasilia,2023-01-01 04:00:00+00:00,6.8,5.6,4.1,7.1
5,Brasilia,2023-01-01 05:00:00+00:00,5.6,5.1,3.3,7.9
6,Brasilia,2023-01-01 06:00:00+00:00,5.1,4.9,0.2,8.9
7,Brasilia,2023-01-01 07:00:00+00:00,4.9,4.9,1.0,9.1
8,Brasilia,2023-01-01 08:00:00+00:00,4.9,4.9,1.3,8.9
9,Brasilia,2023-01-01 09:00:00+00:00,4.9,4.1,1.4,9.0



Beijing Air Quality Targets:


,station,Date,PM2.5,PM25_1hr_ahead,PM25_6hr_ahead,PM25_24hr_ahead
0,Aotizhongxin,2013-03-01 00:00:00,4.0,8.0,3.0,22.0
1,Aotizhongxin,2013-03-01 01:00:00,8.0,7.0,3.0,14.0
2,Aotizhongxin,2013-03-01 02:00:00,7.0,6.0,3.0,13.0
3,Aotizhongxin,2013-03-01 03:00:00,6.0,3.0,3.0,3.0
4,Aotizhongxin,2013-03-01 04:00:00,3.0,5.0,3.0,3.0
5,Aotizhongxin,2013-03-01 05:00:00,5.0,3.0,3.0,9.0
6,Aotizhongxin,2013-03-01 06:00:00,3.0,3.0,3.0,4.0
7,Aotizhongxin,2013-03-01 07:00:00,3.0,3.0,3.0,3.0
8,Aotizhongxin,2013-03-01 08:00:00,3.0,3.0,6.0,3.0
9,Aotizhongxin,2013-03-01 09:00:00,3.0,3.0,8.0,10.0


## Train/Test Split

In [27]:
# define output variables
target_columns = [
    "PM25_1hr_ahead",
    "PM25_6hr_ahead",
    "PM25_24hr_ahead"
]

# display input and output counts
print("Global Input Features:", len(global_features))
print("Beijing Input Features:", len(beijing_features))
print("Output Variables:", len(target_columns))

Global Input Features: 6
Beijing Input Features: 27
Output Variables: 3


In [28]:
# define training/testing cutoff dates
global_cutoff = pd.Timestamp("2023-10-01", tz="UTC")
beijing_cutoff = pd.Timestamp("2016-03-01")

# maximum forecasting hours
max_hours = pd.Timedelta(hours=24)

In [29]:
# sort both datasets by date
global_targets = global_targets.sort_values(
    ["Date", "City"]
).reset_index(drop=True)
beijing_targets = beijing_targets.sort_values(
    ["Date", "station"]
).reset_index(drop=True)

# split Global Air Quality
global_train = global_targets[
    global_targets["Date"] + max_hours < global_cutoff
].copy()
global_test = global_targets[
    global_targets["Date"] >= global_cutoff
].copy()

# split Beijing Air Quality
beijing_train = beijing_targets[
    beijing_targets["Date"] + max_hours < beijing_cutoff
].copy()
beijing_test = beijing_targets[
    beijing_targets["Date"] >= beijing_cutoff
].copy()

# display split sizes
print("Global Training Rows:", len(global_train))
print("Global Testing Rows:", len(global_test))
print("\nBeijing Training Rows:", len(beijing_train))
print("Beijing Testing Rows:", len(beijing_test))

Global Training Rows: 39168
Global Testing Rows: 13104

Beijing Training Rows: 253140
Beijing Testing Rows: 86342


In [30]:
# blobal air quality inputs and outputs
X_global_train = global_train[global_features].copy()
X_global_test = global_test[global_features].copy()

Y_global_train = global_train[target_columns].copy()
Y_global_test = global_test[target_columns].copy()

# beijing air quality inputs and outputs
X_beijing_train = beijing_train[beijing_features].copy()
X_beijing_test = beijing_test[beijing_features].copy()

Y_beijing_train = beijing_train[target_columns].copy()
Y_beijing_test = beijing_test[target_columns].copy()

In [31]:
# preserve global location information
global_train_info = global_train[["City", "Date"]].copy()
global_test_info = global_test[["City", "Date"]].copy()

# preserve beijing station information
beijing_train_info = beijing_train[["station", "Date"]].copy()
beijing_test_info = beijing_test[["station", "Date"]].copy()

In [32]:
import numpy as np

def add_lag_and_time_features(targets_df, base_df, loc_col):
    """
    Adds PM2.5 lags (1-3 h), a rolling 24 h mean, and cyclical hour/month features.

    Lags are matched by TIMESTAMP within each location (not .shift()), because rows with
    missing values were dropped earlier, so row order != hourly order.
    Only past/current information is used, so there is no leakage from the future.
    """
    out = targets_df.copy()
    base = (base_df[[loc_col, "Date", "PM2.5"]]
            .sort_values([loc_col, "Date"]).reset_index(drop=True))

    # past PM2.5 values: value at (t - k hours) attached to row t
    for k in [1, 2, 3]:
        lag = base.copy()
        lag["Date"] = lag["Date"] + pd.Timedelta(hours=k)
        lag = lag.rename(columns={"PM2.5": f"PM25_lag{k}"})
        out = out.merge(lag, on=[loc_col, "Date"], how="left", validate="one_to_one")

    # rolling 24 h mean over the window (t-24h, t]; needs >= 12 observations in the window
    roll = (base.set_index("Date")
                .groupby(loc_col)["PM2.5"]
                .rolling("24h", min_periods=12).mean()
                .rename("PM25_roll24").reset_index())
    out = out.merge(roll, on=[loc_col, "Date"], how="left", validate="one_to_one")

    # cyclical time features (so hour 23 is close to hour 0, December close to January)
    hour = out["Date"].dt.hour
    month = out["Date"].dt.month
    out["hour_sin"] = np.sin(2 * np.pi * hour / 24)
    out["hour_cos"] = np.cos(2 * np.pi * hour / 24)
    out["month_sin"] = np.sin(2 * np.pi * (month - 1) / 12)
    out["month_cos"] = np.cos(2 * np.pi * (month - 1) / 12)

    new_cols = ["PM25_lag1", "PM25_lag2", "PM25_lag3", "PM25_roll24",
                "hour_sin", "hour_cos", "month_sin", "month_cos"]

    before = len(out)
    out = out.dropna(subset=new_cols).reset_index(drop=True)
    print(f"{loc_col}: dropped {before - len(out)} rows lacking lag history; {len(out)} remain")
    return out, new_cols

# base frames = cleaned data BEFORE targets were created (they hold every valid hour)
global_targets, new_feature_cols = add_lag_and_time_features(global_targets, global_processed, "City")
beijing_targets, _ = add_lag_and_time_features(beijing_targets, beijing_processed, "station")

# extend the feature lists so the X_* exports include the new columns
global_features = global_features + new_feature_cols
beijing_features = beijing_features + new_feature_cols
print("Global features :", len(global_features))
print("Beijing features:", len(beijing_features))

City: dropped 66 rows lacking lag history; 52350 remain
station: dropped 23117 rows lacking lag history; 316626 remain
Global features : 14
Beijing features: 35


In [33]:
# display input/output dimensions
print("GLOBAL AIR QUALITY")
print("X Train:", X_global_train.shape)
print("X Test:", X_global_test.shape)
print("Y Train:", Y_global_train.shape)
print("Y Test:", Y_global_test.shape)
print("\nBEIJING AIR QUALITY")
print("X Train:", X_beijing_train.shape)
print("X Test:", X_beijing_test.shape)
print("Y Train:", Y_beijing_train.shape)
print("Y Test:", Y_beijing_test.shape)

# verify matching observations
assert len(X_global_train) == len(Y_global_train)
assert len(X_global_test) == len(Y_global_test)
assert len(X_beijing_train) == len(Y_beijing_train)
assert len(X_beijing_test) == len(Y_beijing_test)

# verify no missing values
for dataset in [
    X_global_train, X_global_test,
    Y_global_train, Y_global_test,
    X_beijing_train, X_beijing_test,
    Y_beijing_train, Y_beijing_test
]:
    assert not dataset.isnull().any().any()

# verify chronological boundaries
assert global_train["Date"].max() + max_hours < global_cutoff
assert global_test["Date"].min() >= global_cutoff
assert beijing_train["Date"].max() + max_hours < beijing_cutoff
assert beijing_test["Date"].min() >= beijing_cutoff

print("\nAll train/test checks passed!")

GLOBAL AIR QUALITY
X Train: (39168, 6)
X Test: (13104, 6)
Y Train: (39168, 3)
Y Test: (13104, 3)

BEIJING AIR QUALITY
X Train: (253140, 27)
X Test: (86342, 27)
Y Train: (253140, 3)
Y Test: (86342, 3)

All train/test checks passed!


## Export Processed Data

In [34]:
# create folder for processed datasets
os.makedirs("processed", exist_ok=True)

In [35]:
# export global training and testing inputs
X_global_train.to_csv("processed/X_global_train.csv", index=False)
X_global_test.to_csv("processed/X_global_test.csv", index=False)

# export global training and testing outputs
Y_global_train.to_csv("processed/Y_global_train.csv", index=False)
Y_global_test.to_csv("processed/Y_global_test.csv", index=False)

# export global testing location and timestamp information
global_test_info.to_csv("processed/global_test_info.csv", index=False)

In [36]:
# export Beijing training and testing inputs
X_beijing_train.to_csv("processed/X_beijing_train.csv", index=False)
X_beijing_test.to_csv("processed/X_beijing_test.csv", index=False)

# export Beijing training and testing outputs
Y_beijing_train.to_csv("processed/Y_beijing_train.csv", index=False)
Y_beijing_test.to_csv("processed/Y_beijing_test.csv", index=False)

# export Beijing testing location and timestamp information
beijing_test_info.to_csv("processed/beijing_test_info.csv", index=False)

In [37]:
# Training location/timestamp info (rows line up 1:1 with X_*_train / Y_*_train)
global_train_info.to_csv("processed/global_train_info.csv", index=False)
beijing_train_info.to_csv("processed/beijing_train_info.csv", index=False)
print("Exported train info files")

Exported train info files


In [38]:
# display exported files
print("Exported Processed Datasets:")

for file in sorted(os.listdir("processed")):
    print(file)

print("\nAll processed datasets exported!")

Exported Processed Datasets:
X_beijing_test.csv
X_beijing_train.csv
X_global_test.csv
X_global_train.csv
Y_beijing_test.csv
Y_beijing_train.csv
Y_global_test.csv
Y_global_train.csv
beijing_test_info.csv
beijing_train_info.csv
global_test_info.csv
global_train_info.csv

All processed datasets exported!
